In [ ]:
import os
import sys
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import nibabel as nib
import pandas as pd
from tqdm import tqdm
from itertools import combinations

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import matplotlib.pyplot as plt
import seaborn as sns
from mpl_toolkits.mplot3d import Axes3D
from matplotlib.patches import FancyBboxPatch, Circle, Patch
from matplotlib.collections import LineCollection
from sklearn.decomposition import PCA

# Seaborn styling for research publications
sns.set_style("whitegrid")
sns.set_context("paper", font_scale=1.3)
plt.rcParams['figure.dpi'] = 300
plt.rcParams['savefig.dpi'] = 300
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['font.size'] = 11
plt.rcParams['axes.labelsize'] = 12
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['xtick.labelsize'] = 10
plt.rcParams['ytick.labelsize'] = 10
plt.rcParams['legend.fontsize'] = 10
plt.rcParams['axes.labelweight'] = 'bold'
plt.rcParams['axes.titleweight'] = 'bold'

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Random seeds
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)

In [ ]:
# Project structure
PROJECT_ROOT = Path('..')  # Go up from notebooks to MedBIND3D
PR_ROOT = PROJECT_ROOT / 'Protocol_Redundancy'

# Data path
DATA_ROOT = PROJECT_ROOT / 'BraTS2020_ValidationData' / 'MICCAI_BraTS2020_ValidationData'

# Output directories
RESULTS_DIR = PR_ROOT / 'results'
VIZ_DIR = RESULTS_DIR / 'visualizations'
SCORES_DIR = RESULTS_DIR / 'scores'
MODELS_DIR = RESULTS_DIR / 'models'

# Create ALL directories with absolute path resolution
print("\n" + "="*80)
print("CREATING DIRECTORY STRUCTURE")
print("="*80)

for directory in [PR_ROOT, RESULTS_DIR, VIZ_DIR, SCORES_DIR, MODELS_DIR]:
    abs_path = directory.resolve()
    abs_path.mkdir(parents=True, exist_ok=True)
    print(f"✓ Created: {abs_path}")

print("="*80)

# Modalities
MODALITIES = ['flair', 't1', 't1ce', 't2']
MODALITY_NAMES = {'flair': 'FLAIR', 't1': 'T1', 't1ce': 'T1CE', 't2': 'T2'}
MODALITY_PAIRS = list(combinations(MODALITIES, 2))

# Hyperparameters - IMPROVED
PATCH_SIZE = 64
LATENT_DIM = 32
BATCH_SIZE = 16  # Increased
NUM_EPOCHS_DCCA = 50  # Increased from 20
LEARNING_RATE = 1e-4
NUM_PATIENTS = 100  # Increased from 30
PATCHES_PER_VOLUME = 10

print(f"\n📊 Training Configuration:")
print(f"  Patients: {NUM_PATIENTS}")
print(f"  Patches per volume: {PATCHES_PER_VOLUME}")
print(f"  Total samples: ~{NUM_PATIENTS * PATCHES_PER_VOLUME}")
print(f"  DCCA Epochs: {NUM_EPOCHS_DCCA}")
print(f"  Batch Size: {BATCH_SIZE}")
print(f"\n🔬 Modality Pairs: {len(MODALITY_PAIRS)}")
for pair in MODALITY_PAIRS:
    print(f"  {MODALITY_NAMES[pair[0]]} ↔ {MODALITY_NAMES[pair[1]]}")

In [ ]:
class BraTS2020MultimodalDataset(Dataset):
    """BraTS2020 Dataset with per-modality global normalization"""
    
    def __init__(self, data_root, modalities, patch_size=64, 
                 num_patients=100, num_patches_per_volume=10):
        self.data_root = Path(data_root)
        self.modalities = modalities
        self.patch_size = patch_size
        self.num_patches_per_volume = num_patches_per_volume
        
        # Find all patient directories
        self.patient_dirs = sorted([d for d in self.data_root.iterdir() if d.is_dir()])
        self.patient_dirs = self.patient_dirs[:num_patients]
        print(f"Found {len(self.patient_dirs)} patients (using {num_patients})")
        
        # STEP 1: Compute global normalization statistics per modality
        print("\n" + "="*60)
        print("COMPUTING GLOBAL NORMALIZATION STATISTICS")
        print("="*60)
        
        self.modality_stats = {}
        for mod in tqdm(modalities, desc="Computing stats"):
            all_values = []
            for patient_dir in self.patient_dirs[:20]:  # Use subset for stats
                patient_id = patient_dir.name
                file_path = patient_dir / f"{patient_id}_{mod}.nii"
                if file_path.exists():
                    vol = nib.load(str(file_path)).get_fdata()
                    # Sample random voxels
                    mask = vol > vol.mean() * 0.1  # Brain region
                    brain_voxels = vol[mask]
                    if len(brain_voxels) > 0:
                        sample = np.random.choice(brain_voxels, 
                                                  size=min(10000, len(brain_voxels)), 
                                                  replace=False)
                        all_values.extend(sample)
            
            all_values = np.array(all_values)
            self.modality_stats[mod] = {
                'mean': np.mean(all_values),
                'std': np.std(all_values)
            }
            print(f"{MODALITY_NAMES[mod]:6s}: mean={self.modality_stats[mod]['mean']:.2f}, "
                  f"std={self.modality_stats[mod]['std']:.2f}")
        
        print("="*60)
        
        # STEP 2: Extract patches with global normalization
        self.patches = {mod: [] for mod in modalities}
        self.patient_ids = []
        
        print("\nExtracting patches with global normalization...")
        for patient_dir in tqdm(self.patient_dirs, desc="Loading patients"):
            patient_id = patient_dir.name
            
            # Load all modalities
            volumes = {}
            for mod in self.modalities:
                file_path = patient_dir / f"{patient_id}_{mod}.nii"
                if file_path.exists():
                    vol = nib.load(str(file_path)).get_fdata()
                    # Global normalization using pre-computed stats
                    vol = (vol - self.modality_stats[mod]['mean']) / \
                          (self.modality_stats[mod]['std'] + 1e-8)
                    volumes[mod] = vol
            
            if len(volumes) == len(self.modalities):
                vol_shape = list(volumes.values())[0].shape
                
                for _ in range(self.num_patches_per_volume):
                    # Random location
                    x = np.random.randint(20, vol_shape[0] - self.patch_size - 20)
                    y = np.random.randint(20, vol_shape[1] - self.patch_size - 20)
                    z = np.random.randint(20, vol_shape[2] - self.patch_size - 20)
                    
                    # Extract patches
                    patches_ok = True
                    temp_patches = {}
                    for mod in self.modalities:
                        patch = volumes[mod][x:x+self.patch_size, 
                                            y:y+self.patch_size, 
                                            z:z+self.patch_size]
                        if patch.std() > 0.1:
                            temp_patches[mod] = patch.copy()
                        else:
                            patches_ok = False
                            break
                    
                    if patches_ok:
                        for mod in self.modalities:
                            self.patches[mod].append(temp_patches[mod])
                        self.patient_ids.append(patient_id)
        
        # Convert to arrays
        for mod in self.modalities:
            self.patches[mod] = np.array(self.patches[mod])
        
        print(f"\n✓ Total patches extracted: {len(self.patient_ids)}")
        print(f"✓ Patch shape: {self.patches[self.modalities[0]][0].shape}")
        
    def __len__(self):
        return len(self.patient_ids)
    
    def __getitem__(self, idx):
        sample = {'patient_id': self.patient_ids[idx]}
        for mod in self.modalities:
            patch = self.patches[mod][idx]
            sample[mod] = torch.FloatTensor(patch).unsqueeze(0)
        return sample

In [ ]:
# Create dataset and dataloader
dataset = BraTS2020MultimodalDataset(
    data_root=DATA_ROOT,
    modalities=MODALITIES,
    patch_size=PATCH_SIZE,
    num_patients=NUM_PATIENTS,
    num_patches_per_volume=PATCHES_PER_VOLUME
)

dataloader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0
)

print(f"\n✓ DataLoader created with {len(dataloader)} batches")

In [ ]:
class DCCAEncoder(nn.Module):
    """3D CNN encoder for DCCA"""
    
    def __init__(self, input_channels=1, latent_dim=32):
        super().__init__()
        
        self.encoder = nn.Sequential(
            nn.Conv3d(input_channels, 32, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm3d(32),
            nn.ReLU(inplace=True),
            
            nn.Conv3d(32, 64, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm3d(64),
            nn.ReLU(inplace=True),
            
            nn.Conv3d(64, 128, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm3d(128),
            nn.ReLU(inplace=True),
            
            nn.Conv3d(128, 256, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm3d(256),
            nn.ReLU(inplace=True),
            
            nn.AdaptiveAvgPool3d(1)
        )
        
        self.fc = nn.Linear(256, latent_dim)
        
    def forward(self, x):
        features = self.encoder(x)
        features = features.view(features.size(0), -1)
        latent = self.fc(features)
        return latent

class DCCAModel(nn.Module):
    """DCCA model for learning correlated representations"""
    
    def __init__(self, latent_dim=32):
        super().__init__()
        self.encoder1 = DCCAEncoder(input_channels=1, latent_dim=latent_dim)
        self.encoder2 = DCCAEncoder(input_channels=1, latent_dim=latent_dim)
        
    def forward(self, x1, x2):
        z1 = self.encoder1(x1)
        z2 = self.encoder2(x2)
        return z1, z2

def dcca_loss(z1, z2, r1=1e-3, r2=1e-3):
    """Deep Canonical Correlation Analysis loss"""
    batch_size = z1.shape[0]
    latent_dim = z1.shape[1]
    
    # Center
    z1_centered = z1 - z1.mean(dim=0, keepdim=True)
    z2_centered = z2 - z2.mean(dim=0, keepdim=True)
    
    # Covariance matrices
    C11 = (z1_centered.t() @ z1_centered) / (batch_size - 1)
    C22 = (z2_centered.t() @ z2_centered) / (batch_size - 1)
    C12 = (z1_centered.t() @ z2_centered) / (batch_size - 1)
    
    # Regularization
    C11 = C11 + r1 * torch.eye(latent_dim, device=z1.device)
    C22 = C22 + r2 * torch.eye(latent_dim, device=z2.device)
    
    try:
        C11_inv_sqrt = torch.linalg.inv(torch.linalg.cholesky(C11))
        C22_inv_sqrt = torch.linalg.inv(torch.linalg.cholesky(C22))
        T = C11_inv_sqrt @ C12 @ C22_inv_sqrt.t()
        corr = torch.trace(T @ T.t()).sqrt()
        loss = -corr
    except:
        loss = -torch.norm(C12, p='fro')
    
    return loss

In [ ]:
# Train DCCA for each modality pair
dcca_models = {}
dcca_correlations = {}

print("\n" + "="*80)
print("TRAINING DCCA MODELS (50 EPOCHS EACH)")
print("="*80)

for mod1, mod2 in MODALITY_PAIRS:
    pair_name = f"{MODALITY_NAMES[mod1]}-{MODALITY_NAMES[mod2]}"
    print(f"\n{'='*60}")
    print(f"Training: {pair_name}")
    print(f"{'='*60}")
    
    model = DCCAModel(latent_dim=LATENT_DIM).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-5)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS_DCCA)
    
    model.train()
    for epoch in range(NUM_EPOCHS_DCCA):
        total_loss = 0
        for batch in dataloader:
            x1 = batch[mod1].to(device)
            x2 = batch[mod2].to(device)
            
            optimizer.zero_grad()
            z1, z2 = model(x1, x2)
            loss = dcca_loss(z1, z2)
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
        
        scheduler.step()
        avg_loss = total_loss / len(dataloader)
        
        if (epoch + 1) % 10 == 0:
            print(f"Epoch {epoch+1:02d}/{NUM_EPOCHS_DCCA} | Loss: {avg_loss:.4f} | "
                  f"LR: {scheduler.get_last_lr()[0]:.6f}")
    
    # Compute final correlation
    model.eval()
    all_z1, all_z2 = [], []
    
    with torch.no_grad():
        for batch in dataloader:
            x1 = batch[mod1].to(device)
            x2 = batch[mod2].to(device)
            z1, z2 = model(x1, x2)
            all_z1.append(z1.cpu())
            all_z2.append(z2.cpu())
    
    all_z1 = torch.cat(all_z1, dim=0)
    all_z2 = torch.cat(all_z2, dim=0)
    
    # Correlation
    correlation = torch.corrcoef(torch.cat([all_z1, all_z2], dim=1).t())
    mean_corr = correlation[:LATENT_DIM, LATENT_DIM:].abs().mean().item()
    
    dcca_models[(mod1, mod2)] = model
    dcca_correlations[(mod1, mod2)] = mean_corr
    
    print(f"\n✓ Final DCCA correlation: {mean_corr:.4f}")
    
    # Save model
    model_path = MODELS_DIR / f'dcca_{mod1}_{mod2}.pth'
    torch.save(model.state_dict(), model_path)
    print(f"✓ Saved: {model_path}")

print("\n" + "="*80)
print("DCCA TRAINING COMPLETE")
print("="*80)

In [ ]:
def rbf_kernel(X, sigma=1.0):
    """RBF kernel"""
    XX = torch.sum(X**2, dim=1, keepdim=True)
    distances = XX + XX.t() - 2 * torch.mm(X, X.t())
    K = torch.exp(-distances / (2 * sigma**2))
    return K

def center_kernel_matrix(K):
    """Center kernel matrix"""
    n = K.shape[0]
    H = torch.eye(n, device=K.device) - torch.ones(n, n, device=K.device) / n
    return H @ K @ H

def hsic_score(X, Y, sigma_x=1.0, sigma_y=1.0):
    """HSIC score"""
    n = X.shape[0]
    K_x = rbf_kernel(X, sigma=sigma_x)
    K_y = rbf_kernel(Y, sigma=sigma_y)
    K_x_centered = center_kernel_matrix(K_x)
    K_y_centered = center_kernel_matrix(K_y)
    hsic = torch.trace(K_x_centered @ K_y_centered) / ((n - 1) ** 2)
    return hsic.item()

In [ ]:
# Compute HSIC with PCA
hsic_scores = {}

print("\n" + "="*80)
print("COMPUTING HSIC SCORES WITH PCA")
print("="*80)

# Extract features
all_features = {mod: [] for mod in MODALITIES}

print("\nExtracting features...")
for batch in tqdm(dataloader, desc="Processing batches"):
    for mod in MODALITIES:
        x = batch[mod].to(device)
        features = x.view(x.shape[0], -1)
        all_features[mod].append(features.cpu())

for mod in MODALITIES:
    all_features[mod] = torch.cat(all_features[mod], dim=0)
    print(f"{MODALITY_NAMES[mod]:6s}: {all_features[mod].shape}")

# Apply PCA to reduce dimensionality
print("\nApplying PCA (262144 → 100 dimensions)...")
pca_models = {}
features_pca = {}

for mod in tqdm(MODALITIES, desc="PCA"):
    pca = PCA(n_components=100, random_state=42)
    features_pca[mod] = pca.fit_transform(all_features[mod].numpy())
    pca_models[mod] = pca
    print(f"{MODALITY_NAMES[mod]:6s}: Variance explained = {pca.explained_variance_ratio_.sum():.3f}")

# Compute HSIC
print("\nComputing HSIC scores...")
for mod1, mod2 in tqdm(MODALITY_PAIRS, desc="HSIC"):
    X = torch.FloatTensor(features_pca[mod1])
    Y = torch.FloatTensor(features_pca[mod2])
    
    # Use subset for efficiency
    n_samples = min(300, X.shape[0])
    indices = torch.randperm(X.shape[0])[:n_samples]
    X_subset = X[indices]
    Y_subset = Y[indices]
    
    # Median heuristic
    sigma_x = torch.median(torch.pdist(X_subset)).item()
    sigma_y = torch.median(torch.pdist(Y_subset)).item()
    
    hsic = hsic_score(X_subset, Y_subset, sigma_x=sigma_x, sigma_y=sigma_y)
    hsic_scores[(mod1, mod2)] = hsic
    
    print(f"{MODALITY_NAMES[mod1]:6s}-{MODALITY_NAMES[mod2]:6s}: HSIC = {hsic:.6f}")

print("\n" + "="*80)
print("HSIC COMPUTATION COMPLETE")
print("="*80)

In [ ]:
# Normalize and combine
dcca_values = np.array(list(dcca_correlations.values()))
hsic_values = np.array(list(hsic_scores.values()))

dcca_normalized = (dcca_values - dcca_values.min()) / (dcca_values.max() - dcca_values.min() + 1e-8)
hsic_normalized = (hsic_values - hsic_values.min()) / (hsic_values.max() - hsic_values.min() + 1e-8)

combined_redundancy = {}
for i, (mod1, mod2) in enumerate(MODALITY_PAIRS):
    combined = (dcca_normalized[i] + hsic_normalized[i]) / 2
    combined_redundancy[(mod1, mod2)] = combined

# Create dataframe
redundancy_df = pd.DataFrame([
    {
        'Modality_1': MODALITY_NAMES[mod1],
        'Modality_2': MODALITY_NAMES[mod2],
        'DCCA_Correlation': dcca_correlations[(mod1, mod2)],
        'HSIC_Score': hsic_scores[(mod1, mod2)],
        'Combined_Redundancy': combined_redundancy[(mod1, mod2)]
    }
    for mod1, mod2 in MODALITY_PAIRS
])

redundancy_df = redundancy_df.sort_values('Combined_Redundancy', ascending=False)

print("\n" + "="*80)
print("PROTOCOL REDUNDANCY ANALYSIS RESULTS")
print("="*80)
print(redundancy_df.to_string(index=False))
print("="*80)

# Save
csv_path = SCORES_DIR / 'redundancy_scores.csv'
redundancy_df.to_csv(csv_path, index=False)
print(f"\n✓ Saved: {csv_path}")

In [ ]:
# Iceberg plot
sorted_pairs = sorted(combined_redundancy.items(), key=lambda x: x[1], reverse=True)
colors = sns.color_palette("Blues_r", n_colors=len(sorted_pairs))

fig, ax = plt.subplots(figsize=(14, 8))
y_positions = np.arange(len(sorted_pairs))
layer_heights = []

for i, ((mod1, mod2), redundancy) in enumerate(sorted_pairs):
    shared = redundancy * 100
    unique = (1 - redundancy) * 100
    
    ax.barh(y_positions[i], shared, left=0, height=0.7, 
            color=colors[i], edgecolor='white', linewidth=2)
    ax.barh(y_positions[i], unique, left=shared, height=0.7,
            color='lightgray', edgecolor='white', linewidth=2, alpha=0.5)
    layer_heights.append(shared)

waterline = np.median(layer_heights)
ax.axvline(waterline, color='steelblue', linewidth=3, linestyle='--', alpha=0.7)

pair_labels = [f"{MODALITY_NAMES[mod1]}–{MODALITY_NAMES[mod2]}" 
               for (mod1, mod2), _ in sorted_pairs]
ax.set_yticks(y_positions)
ax.set_yticklabels(pair_labels, fontsize=11)
ax.set_xlabel('Information Content (%)', fontsize=12, fontweight='bold')
ax.set_title('Redundancy Iceberg Plot\nShared vs Unique Information', 
             fontsize=14, fontweight='bold', pad=20)

legend_elements = [
    Patch(facecolor='steelblue', label='Shared Information'),
    Patch(facecolor='lightgray', alpha=0.5, label='Unique Information')
]
ax.legend(handles=legend_elements, loc='lower right', frameon=True)
ax.set_xlim(0, 100)
ax.grid(axis='x', alpha=0.3)
sns.despine()

plt.tight_layout()
save_path = VIZ_DIR / '01_redundancy_iceberg.png'
plt.savefig(save_path, dpi=300, bbox_inches='tight')
plt.show()
print(f"✓ Saved: {save_path}")

In [ ]:
fig = plt.figure(figsize=(14, 12))
ax = fig.add_subplot(111, projection='3d')

vertices = np.array([
    [0, 0, 0],
    [1, 0, 0],
    [0.5, np.sqrt(3)/2, 0],
    [0.5, np.sqrt(3)/6, np.sqrt(2/3)]
])

modality_order = ['flair', 't1', 't1ce', 't2']
colors_mod = {'flair': '#3498db', 't1': '#e74c3c', 't1ce': '#2ecc71', 't2': '#f39c12'}

for i, mod in enumerate(modality_order):
    ax.scatter(*vertices[i], s=500, c=colors_mod[mod], 
               edgecolors='white', linewidths=3, alpha=0.9, depthshade=False)
    ax.text(vertices[i, 0], vertices[i, 1], vertices[i, 2] + 0.12, 
            MODALITY_NAMES[mod], fontsize=14, fontweight='bold', ha='center')

for (mod1, mod2), redundancy in combined_redundancy.items():
    i = modality_order.index(mod1)
    j = modality_order.index(mod2)
    linewidth = 2 + redundancy * 10
    
    ax.plot([vertices[i, 0], vertices[j, 0]],
            [vertices[i, 1], vertices[j, 1]],
            [vertices[i, 2], vertices[j, 2]],
            color='gray', linewidth=linewidth, alpha=0.6)
    
    mid = (vertices[i] + vertices[j]) / 2
    ax.text(mid[0], mid[1], mid[2], f'{redundancy:.2f}',
            fontsize=10, ha='center', 
            bbox=dict(boxstyle='round,pad=0.4', facecolor='white', 
                     edgecolor='gray', alpha=0.9))

ax.set_xlabel('X', fontsize=11)
ax.set_ylabel('Y', fontsize=11)
ax.set_zlabel('Z', fontsize=11)
ax.set_title('Modality Tetrahedron Map\nEdge Thickness = Redundancy', 
             fontsize=14, fontweight='bold', pad=25)
ax.view_init(elev=20, azim=45)
ax.grid(False)
ax.xaxis.pane.fill = False
ax.yaxis.pane.fill = False
ax.zaxis.pane.fill = False

plt.tight_layout()
save_path = VIZ_DIR / '02_modality_tetrahedron.png'
plt.savefig(save_path, dpi=300, bbox_inches='tight')
plt.show()
print(f"✓ Saved: {save_path}")

In [ ]:
sample_data = dataset[0]
slice_idx = PATCH_SIZE // 2
grid_size = 10
step = PATCH_SIZE // grid_size
redundancy_grid = np.zeros((grid_size, grid_size))

for i in range(grid_size):
    for j in range(grid_size):
        x_start, y_start = i * step, j * step
        x_end, y_end = x_start + step, y_start + step
        
        local_redundancies = []
        for mod1, mod2 in MODALITY_PAIRS:
            patch1 = sample_data[mod1][0, x_start:x_end, y_start:y_end, slice_idx]
            patch2 = sample_data[mod2][0, x_start:x_end, y_start:y_end, slice_idx]
            corr = np.corrcoef(patch1.flatten(), patch2.flatten())[0, 1]
            local_redundancies.append(abs(corr))
        
        redundancy_grid[i, j] = np.mean(local_redundancies)

grad_y, grad_x = np.gradient(redundancy_grid)

fig, ax = plt.subplots(figsize=(13, 11))
im = ax.imshow(redundancy_grid.T, cmap='RdYlBu_r', origin='lower', 
               extent=[0, PATCH_SIZE, 0, PATCH_SIZE], alpha=0.7)

X, Y = np.meshgrid(np.linspace(step/2, PATCH_SIZE-step/2, grid_size),
                   np.linspace(step/2, PATCH_SIZE-step/2, grid_size))

magnitude = np.sqrt(grad_x**2 + grad_y**2)
grad_x_norm = grad_x / (magnitude + 1e-6)
grad_y_norm = grad_y / (magnitude + 1e-6)

ax.quiver(X, Y, grad_x_norm.T, grad_y_norm.T, magnitude.T,
          cmap='viridis', scale=12, width=0.005, alpha=0.85,
          headwidth=5, headlength=6)

cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
cbar.set_label('Local Redundancy', fontsize=12, fontweight='bold')

ax.set_xlabel('X Position (voxels)', fontsize=12, fontweight='bold')
ax.set_ylabel('Y Position (voxels)', fontsize=12, fontweight='bold')
ax.set_title('Curl-Field Vector Map\nRedundancy Gradient Flow', 
             fontsize=14, fontweight='bold', pad=20)
ax.set_aspect('equal')

plt.tight_layout()
save_path = VIZ_DIR / '03_curl_field_vector_map.png'
plt.savefig(save_path, dpi=300, bbox_inches='tight')
plt.show()
print(f"✓ Saved: {save_path}")

In [ ]:
fig, ax = plt.subplots(figsize=(13, 13), subplot_kw=dict(projection='polar'))

n_modalities = len(MODALITIES)
angles = np.linspace(0, 2 * np.pi, n_modalities, endpoint=False)
modality_positions = {mod: angles[i] for i, mod in enumerate(MODALITIES)}
colors_chord = {'flair': '#3498db', 't1': '#e74c3c', 't1ce': '#2ecc71', 't2': '#f39c12'}

for mod, angle in modality_positions.items():
    ax.scatter(angle, 1.0, s=1000, c=colors_chord[mod], 
               edgecolors='white', linewidths=4, alpha=0.9, zorder=10)
    
    label_angle = angle
    ha = 'left' if 0 <= label_angle <= np.pi else 'right'
    ax.text(label_angle, 1.18, MODALITY_NAMES[mod], 
            fontsize=16, fontweight='bold', ha=ha, va='center')

for (mod1, mod2), redundancy in combined_redundancy.items():
    angle1 = modality_positions[mod1]
    angle2 = modality_positions[mod2]
    theta = np.linspace(angle1, angle2, 100)
    r = 1.0 - 0.5 * redundancy * np.sin(np.linspace(0, np.pi, 100))
    
    color = plt.cm.plasma(redundancy)
    linewidth = 3 + redundancy * 8
    ax.plot(theta, r, color=color, linewidth=linewidth, alpha=0.7, zorder=5)

ax.set_ylim(0, 1.25)
ax.set_theta_zero_location('N')
ax.set_theta_direction(-1)
ax.axis('off')
ax.set_title('Chord-Galaxy Graph\nRedundancy Network', 
             fontsize=15, fontweight='bold', pad=35, y=1.08)

sm = plt.cm.ScalarMappable(cmap='plasma', norm=plt.Normalize(vmin=0, vmax=1))
sm.set_array([])
cbar = plt.colorbar(sm, ax=ax, fraction=0.03, pad=0.1, orientation='horizontal')
cbar.set_label('Redundancy Strength', fontsize=12, fontweight='bold')

plt.tight_layout()
save_path = VIZ_DIR / '04_chord_galaxy_graph.png'
plt.savefig(save_path, dpi=300, bbox_inches='tight')
plt.show()
print(f"✓ Saved: {save_path}")

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 11))
axes = axes.flatten()

for idx, (mod1, mod2) in enumerate(MODALITY_PAIRS):
    ax = axes[idx]
    grid_size = 16
    step = PATCH_SIZE // grid_size
    local_redundancy = np.zeros((grid_size, grid_size))
    
    for i in range(grid_size):
        for j in range(grid_size):
            x_start, y_start = i * step, j * step
            x_end, y_end = x_start + step, y_start + step
            
            patch1 = sample_data[mod1][0, x_start:x_end, y_start:y_end, slice_idx]
            patch2 = sample_data[mod2][0, x_start:x_end, y_start:y_end, slice_idx]
            
            if patch1.std() > 0.01 and patch2.std() > 0.01:
                corr = np.corrcoef(patch1.flatten(), patch2.flatten())[0, 1]
                local_redundancy[i, j] = abs(corr)
    
    im = ax.imshow(local_redundancy.T, cmap='YlOrRd', origin='lower',
                   extent=[0, PATCH_SIZE, 0, PATCH_SIZE], 
                   vmin=0, vmax=1, interpolation='nearest')
    
    for i in range(grid_size + 1):
        ax.axhline(i * step, color='white', linewidth=1.2, alpha=0.6)
        ax.axvline(i * step, color='white', linewidth=1.2, alpha=0.6)
    
    ax.set_title(f'{MODALITY_NAMES[mod1]} ↔ {MODALITY_NAMES[mod2]}', 
                 fontsize=12, fontweight='bold', pad=10)
    ax.set_xlabel('X', fontsize=10, fontweight='bold')
    ax.set_ylabel('Y', fontsize=10, fontweight='bold')
    ax.set_aspect('equal')
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

plt.suptitle('Redundancy Jigsaw Map\nLocal Block-wise Patterns', 
             fontsize=15, fontweight='bold', y=0.995)
plt.tight_layout()
save_path = VIZ_DIR / '05_redundancy_jigsaw_map.png'
plt.savefig(save_path, dpi=300, bbox_inches='tight')
plt.show()
print(f"✓ Saved: {save_path}")

In [ ]:
# Create matrices
redundancy_matrix_dcca = np.zeros((len(MODALITIES), len(MODALITIES)))
redundancy_matrix_hsic = np.zeros((len(MODALITIES), len(MODALITIES)))
redundancy_matrix_combined = np.zeros((len(MODALITIES), len(MODALITIES)))

for (mod1, mod2), dcca_corr in dcca_correlations.items():
    i = MODALITIES.index(mod1)
    j = MODALITIES.index(mod2)
    
    redundancy_matrix_dcca[i, j] = dcca_corr
    redundancy_matrix_dcca[j, i] = dcca_corr
    
    redundancy_matrix_hsic[i, j] = hsic_scores[(mod1, mod2)]
    redundancy_matrix_hsic[j, i] = hsic_scores[(mod1, mod2)]
    
    redundancy_matrix_combined[i, j] = combined_redundancy[(mod1, mod2)]
    redundancy_matrix_combined[j, i] = combined_redundancy[(mod1, mod2)]

np.fill_diagonal(redundancy_matrix_dcca, 1.0)
np.fill_diagonal(redundancy_matrix_hsic, 1.0)
np.fill_diagonal(redundancy_matrix_combined, 1.0)

modality_labels = [MODALITY_NAMES[mod] for mod in MODALITIES]

pd.DataFrame(redundancy_matrix_dcca, 
             index=modality_labels, 
             columns=modality_labels).to_csv(SCORES_DIR / 'redundancy_matrix_dcca.csv')

pd.DataFrame(redundancy_matrix_hsic, 
             index=modality_labels, 
             columns=modality_labels).to_csv(SCORES_DIR / 'redundancy_matrix_hsic.csv')

pd.DataFrame(redundancy_matrix_combined, 
             index=modality_labels, 
             columns=modality_labels).to_csv(SCORES_DIR / 'redundancy_matrix_combined.csv')

print("\n✓ Saved all redundancy matrices")